# 01 — Разведочный анализ GPS-треков парусных регат

Ноутбук исследует характеристики реальных GPS-треков парусных регат,
использующихся для дальнейшего сравнения методов интерполяции.

**Цели:**
- убедиться в качестве данных после очистки парсером,
- оценить разреженность и характер шума GPS-сигнала,
- отобрать треки для финального исследования,
- сгенерировать рисунок 2 ВКР (иллюстрация GPS-шума).

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter1d

from interp_research.geo import LocalProjection
from interp_research.parser import load_gpx
from interp_research.plotting import THESIS_COLORS, set_thesis_style

set_thesis_style()

In [ ]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
FIGURES_DIR = Path("../results/figures")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Загрузка и обзор всех GPX-файлов

In [ ]:
all_tracks: dict[str, dict[str, np.ndarray]] = {}

rows = []
for gpx_path in sorted(RAW_DIR.glob("*.gpx")):
    tracks = load_gpx(gpx_path)
    all_tracks[gpx_path.stem] = tracks
    for yacht_name, arr in tracks.items():
        duration_s = arr[-1, 0] - arr[0, 0]
        n_points = len(arr)
        avg_freq = n_points / duration_s if duration_s > 0 else 0
        rows.append({
            "Файл": gpx_path.name,
            "Яхта": yacht_name,
            "Точек": n_points,
            "Длительность (мин)": round(duration_s / 60, 1),
            "Ср. частота (Гц)": round(avg_freq, 2),
        })

summary_df = pd.DataFrame(rows)
summary_df

Треки длятся от 30 до 300+ минут и содержат от 300 до 4000 точек. Частота записи варьируется от ~0.1 Гц (10 с) до ~0.3 Гц (3 с) в зависимости от трекера — достаточно данных для экспериментов с прореживанием и интерполяцией.

## Визуализация одной регаты на карте

Выбираем регату с несколькими яхтами и отображаем их траектории в локальной проекции.

In [ ]:
# Выбираем регату с наибольшим числом яхт; при равенстве — первую по алфавиту
regatta_name = max(all_tracks, key=lambda k: len(all_tracks[k]))
regatta = all_tracks[regatta_name]

# Опорная точка — среднее по всем точкам регаты
all_lats = np.concatenate([arr[:, 1] for arr in regatta.values()])
all_lons = np.concatenate([arr[:, 2] for arr in regatta.values()])
proj = LocalProjection(float(all_lats.mean()), float(all_lons.mean()))

fig, ax = plt.subplots()
for i, (yacht, arr) in enumerate(regatta.items()):
    x, y = proj.to_local(arr[:, 1], arr[:, 2])
    color = THESIS_COLORS[i % len(THESIS_COLORS)]
    ax.plot(x, y, color=color, linewidth=0.8, label=yacht)
    ax.plot(x[0], y[0], "o", color=color, markersize=5)

ax.set_xlabel("x, м (восток)")
ax.set_ylabel("y, м (север)")
ax.set_title(f"Регата: {regatta_name}")
ax.set_aspect("equal")
ax.legend(fontsize=8)
fig.suptitle("Рисунок: типичный набор треков парусной регаты, локальная проекция, метры",
             fontsize=9, y=-0.02)
plt.tight_layout()
plt.show()

## Анализ частоты записи

Гистограмма интервалов $\Delta t$ между соседними GPS-точками по всем трекам.

In [ ]:
all_dt = []
for file_tracks in all_tracks.values():
    for arr in file_tracks.values():
        dt = np.diff(arr[:, 0])
        all_dt.append(dt)

all_dt = np.concatenate(all_dt)

# Для гистограммы отбрасываем гигантские межсегментные разрывы (> 5 мин)
dt_regular = all_dt[all_dt <= 300]
n_gaps = np.sum(all_dt > 300)

fig, ax = plt.subplots()
ax.hist(dt_regular, bins=np.arange(0, 32, 1), edgecolor="white", linewidth=0.3)
ax.set_xlabel("Интервал между точками, с")
ax.set_ylabel("Число интервалов")
ax.set_title("Распределение $\\Delta t$ по всем трекам (без разрывов > 5 мин)")
ax.axvline(np.median(dt_regular), color=THESIS_COLORS[2], linestyle="--",
           label=f"Медиана = {np.median(dt_regular):.0f} с")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Медиана Δt: {np.median(dt_regular):.1f} с")
print(f"Среднее Δt: {np.mean(dt_regular):.1f} с")
print(f"Макс Δt (без разрывов): {np.max(dt_regular):.0f} с")
print(f"Доля Δt ≤ 15 с: {(dt_regular <= 15).mean():.1%}")
print(f"Крупных разрывов (> 5 мин): {n_gaps}")

Типичная частота записи — порядка 0.1–0.3 Гц ($\Delta t$ от 3 до 10 с в зависимости от трекера). В некоторых файлах присутствуют крупные разрывы (многочасовые и даже многодневные) — вероятно, несколько сессий записи объединены в один GPX-трек.

## Анализ GPS-шума (рисунок 2 ВКР)

Визуализация колебаний GPS-координат относительно сглаженной траектории на участке
приблизительно прямолинейного хода. Сглаживание — скользящее среднее (uniform filter).

In [ ]:
# Ищем самый прямолинейный участок среди ВСЕХ треков
SEGMENT_LEN = 60
SMOOTH_WINDOW = 15

best_score = np.inf
best_seg = None
best_label = ""

for file_stem, tracks in all_tracks.items():
    for yacht_name, arr in tracks.items():
        if len(arr) < SEGMENT_LEN + 10:
            continue
        p = LocalProjection(float(arr[:, 1].mean()), float(arr[:, 2].mean()))
        xf, yf = p.to_local(arr[:, 1], arr[:, 2])

        # Отсекаем участки с большими временными разрывами
        dt = np.diff(arr[:, 0])

        brg = np.degrees(np.arctan2(np.diff(xf), np.diff(yf)))
        brg_std = pd.Series(brg).rolling(15, center=True).std().values
        brg_std = np.nan_to_num(brg_std, nan=999.0)

        for s in range(len(brg_std) - SEGMENT_LEN):
            if np.max(dt[s : s + SEGMENT_LEN]) > 30:
                continue
            score = np.mean(brg_std[s : s + SEGMENT_LEN])
            if score < best_score:
                best_score = score
                sl = slice(s, s + SEGMENT_LEN)
                x_seg = xf[sl].copy()
                y_seg = yf[sl].copy()
                best_label = f"{file_stem} / {yacht_name}"

x_smooth = uniform_filter1d(x_seg, size=SMOOTH_WINDOW)
y_smooth = uniform_filter1d(y_seg, size=SMOOTH_WINDOW)

print(f"Лучший участок: {best_label} (σ азимута = {best_score:.1f}°)")

In [ ]:
# Поворачиваем сегмент так, чтобы основное направление было вдоль оси X
dx = x_seg[-1] - x_seg[0]
dy = y_seg[-1] - y_seg[0]
angle = np.arctan2(dy, dx)

cos_a, sin_a = np.cos(-angle), np.sin(-angle)
x_rot = (x_seg - x_seg[0]) * cos_a - (y_seg - y_seg[0]) * sin_a
y_rot = (x_seg - x_seg[0]) * sin_a + (y_seg - y_seg[0]) * cos_a
x_sm_rot = (x_smooth - x_seg[0]) * cos_a - (y_smooth - y_seg[0]) * sin_a
y_sm_rot = (x_smooth - x_seg[0]) * sin_a + (y_smooth - y_seg[0]) * cos_a

deviation = y_rot - y_sm_rot

fig, axes = plt.subplots(2, 1, figsize=(6.3, 5.0), height_ratios=[2, 1])

ax1 = axes[0]
ax1.plot(x_rot, y_rot, ".-", color=THESIS_COLORS[3], markersize=4, linewidth=0.6,
         label="Исходные GPS-точки", zorder=2)
ax1.plot(x_sm_rot, y_sm_rot, "-", color=THESIS_COLORS[0], linewidth=2.0,
         label="Сглаженная траектория", zorder=3)
ax1.set_xlabel("Вдоль трека, м")
ax1.set_ylabel("Поперёк трека, м")
ax1.set_title("Фрагмент GPS-трека: шум вокруг сглаженной линии")
ax1.legend(loc="best")

ax2 = axes[1]
ax2.fill_between(x_rot, deviation, 0, alpha=0.3, color=THESIS_COLORS[3])
ax2.plot(x_rot, deviation, ".-", color=THESIS_COLORS[3], markersize=3, linewidth=0.6)
ax2.axhline(0, color=THESIS_COLORS[0], linewidth=1.0)
ax2.set_xlabel("Вдоль трека, м")
ax2.set_ylabel("Отклонение, м")
ax2.set_title(f"Поперечное отклонение от сглаженной линии "
              f"(σ = {np.std(deviation):.1f} м)")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "fig_02_gps_noise.png", dpi=300)
print(f"Сохранено: {FIGURES_DIR / 'fig_02_gps_noise.png'}")
plt.show()

## Сохранение обработанных данных в `data/processed/`

Для каждого файла-яхты сохраняем `.npz` с массивами `t`, `lat`, `lon`, `x_local`, `y_local`.
Локальная проекция строится с центром в средней точке всех треков данного файла.

In [ ]:
saved_count = 0

for file_stem, tracks in all_tracks.items():
    # Опорная точка — среднее по всем трекам этого файла
    file_lats = np.concatenate([a[:, 1] for a in tracks.values()])
    file_lons = np.concatenate([a[:, 2] for a in tracks.values()])
    file_proj = LocalProjection(float(file_lats.mean()), float(file_lons.mean()))

    for yacht_name, arr in tracks.items():
        t = arr[:, 0]
        lat = arr[:, 1]
        lon = arr[:, 2]
        x_loc, y_loc = file_proj.to_local(lat, lon)

        safe_name = yacht_name.replace(" ", "_").replace("/", "_")
        out_path = PROCESSED_DIR / f"{file_stem}__{safe_name}.npz"
        np.savez(out_path, t=t, lat=lat, lon=lon, x_local=x_loc, y_local=y_loc)
        saved_count += 1

print(f"Сохранено {saved_count} файлов в {PROCESSED_DIR}/")

## Итог

Подготовлено **N** очищенных треков для исследования (точное число выведено ячейкой выше).
Рисунок 2 ВКР (`results/figures/fig_02_gps_noise.png`) сгенерирован — иллюстрирует
характерный GPS-шум на участке прямолинейного хода парусной яхты.